# 1. Project Title and Objective

## Sales Prediction Using Python and Machine Learning

**Objective:** Predict product **Sales** from advertising spend on **TV**, **Radio** and **Newspaper**, compare a baseline **Linear Regression** with a **Random Forest Regressor**, and work out which advertising channels appear most influential.

**Workflow:** load data -> inspect/clean -> explore and visualise -> split 80/20 -> train two models -> evaluate (MAE, RMSE, R²) -> select best model -> residual analysis -> channel impact -> sample prediction -> conclusion.

> **Note on units:** In the classic Advertising dataset (James et al., *An Introduction to Statistical Learning*), advertising budgets are in thousands of dollars and Sales is in thousands of units. This notebook verifies the columns from the file you load and does not assume anything beyond that.

# 2. Import Libraries

In [ ]:
import os
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")      # keep the notebook output clean
sns.set_theme(style="whitegrid")       # consistent, readable plot style
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42                      # one seed used everywhere for reproducibility
np.random.seed(RANDOM_STATE)
print("Libraries imported successfully.")

# 3. Dataset Loading

## How to get `Advertising.csv`
Use **one** of these legitimate sources:

1. **Kaggle:** search for *"Advertising dataset"* (for example the dataset `ashydv/advertising-dataset`), download it and place `Advertising.csv` next to this notebook. Kaggle needs a free account.
2. **ISLR book website:** `Advertising.csv` is distributed with *An Introduction to Statistical Learning* (`https://www.statlearning.com/s/Advertising.csv`).

## File path
Set `DATA_PATH` below. If the file is missing, the cell tries to download it from the ISLR URL; if that also fails (for example no internet), it stops with a clear message. **No data is ever generated or invented.**

If you run on Kaggle/Colab, change `DATA_PATH` to the file's location (for example `/kaggle/input/<dataset-name>/Advertising.csv`).

In [ ]:
DATA_PATH = "Advertising.csv"                                   # <-- change if your file is elsewhere
FALLBACK_URL = "https://www.statlearning.com/s/Advertising.csv"  # official ISLR copy (optional download)

if not os.path.exists(DATA_PATH):
    print(f"'{DATA_PATH}' not found. Trying to download from the ISLR website ...")
    try:
        urllib.request.urlretrieve(FALLBACK_URL, DATA_PATH)
        print("Download complete.")
    except Exception as e:
        raise FileNotFoundError(
            f"Could not find or download the dataset ({e}).\n"
            "Please download Advertising.csv manually (see the markdown above) "
            "and set DATA_PATH to its location."
        )

df_raw = pd.read_csv(DATA_PATH)
print(f"Loaded '{DATA_PATH}' successfully.")
print("Column names found in the file:", list(df_raw.columns))

# 4. Data Cleaning and Inspection

Steps:
1. Strip stray spaces from column names and drop any unnamed index column (the Kaggle version often has `Unnamed: 0`).
2. **Check** that `TV`, `Radio`, `Newspaper` and `Sales` exist (case-insensitive) instead of assuming.
3. Force the four columns to numeric and report missing values and duplicates.
4. Drop rows with a missing **target**. Missing **features** (if any) are imputed *inside the modelling pipeline*, so the imputation values are learned from the training data only (no leakage).

In [ ]:
df = df_raw.copy()

# 1) Tidy column names and drop unnamed index-like columns
df.columns = df.columns.astype(str).str.strip()
unnamed = [c for c in df.columns if c.lower().startswith("unnamed")]
if unnamed:
    print("Dropping index-like column(s):", unnamed)
    df = df.drop(columns=unnamed)

# 2) Verify required columns (case-insensitive) and standardise their names
REQUIRED = ["TV", "Radio", "Newspaper", "Sales"]
lookup = {c.lower(): c for c in df.columns}
missing_cols = [c for c in REQUIRED if c.lower() not in lookup]
if missing_cols:
    raise ValueError(f"Required column(s) {missing_cols} not found. Columns available: {list(df.columns)}")
df = df.rename(columns={lookup[c.lower()]: c for c in REQUIRED})[REQUIRED]
print("Columns identified ->", list(df.columns))

# 3) Make sure every column is numeric
for col in REQUIRED:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("\nMissing values per column:")
print(df.isnull().sum())

n_dups = df.duplicated().sum()
print(f"\nDuplicate rows: {n_dups}")
if n_dups > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Removed duplicates. New shape: {df.shape}")

# 4) A row without a target cannot be used for supervised learning
n_before = len(df)
df = df.dropna(subset=["Sales"]).reset_index(drop=True)
if len(df) < n_before:
    print(f"Dropped {n_before - len(df)} row(s) with missing Sales.")

FEATURES = ["TV", "Radio", "Newspaper"]   # input features (order is fixed and reused everywhere)
TARGET = "Sales"                          # what we predict
print("\nFeatures:", FEATURES, "| Target:", TARGET)

# 5. Exploratory Data Analysis

In [ ]:
print("First five rows:")
display(df.head())

print("Shape (rows, columns):", df.shape)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nDescriptive statistics:")
display(df.describe())

# 6. Data Visualizations

### 6.1 Pairplot of all numerical features
The diagonal shows each variable's distribution and the off-diagonal panels show pairwise relationships.

In [ ]:
sns.pairplot(df, diag_kind="kde", plot_kws={"alpha": 0.6})
plt.suptitle("Pairplot of Advertising Spend and Sales", y=1.02, fontsize=14)
plt.show()

### 6.2 Sales versus each advertising channel
Each plot has a fitted linear trend line with a confidence band.

In [ ]:
for channel in FEATURES:
    plt.figure(figsize=(7, 5))
    sns.regplot(data=df, x=channel, y=TARGET,
                scatter_kws={"alpha": 0.6}, line_kws={"color": "red"})
    plt.title(f"Sales vs. {channel} Advertising Spend")
    plt.xlabel(f"{channel} advertising spend")
    plt.ylabel("Sales")
    plt.tight_layout()
    plt.show()

### 6.3 Annotated correlation heatmap
Pearson correlation measures the strength of the **linear** relationship (-1 to +1).

In [ ]:
corr = df.corr()
plt.figure(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

### 6.4 Interpretation (generated from the actual data)
The cell below reads the real correlation values, so the wording always matches the data you loaded.

In [ ]:
def describe_strength(r):
    a = abs(r)
    if a >= 0.7: return "strong"
    if a >= 0.4: return "moderate"
    if a >= 0.2: return "weak"
    return "very weak / negligible"

corr_with_sales = corr[TARGET].drop(TARGET).sort_values(ascending=False)
print("Correlation of each channel with Sales:\n")
for ch, r in corr_with_sales.items():
    direction = "positive" if r > 0 else "negative"
    print(f"- {ch:<10} r = {r:+.3f}  ->  {describe_strength(r)} {direction} linear relationship")

top = corr_with_sales.abs().idxmax()
print(f"\nThe channel most strongly correlated with Sales is {top}.")

print("\nCorrelation between the advertising channels themselves (relevant for multicollinearity):")
print(corr.loc[FEATURES, FEATURES].round(3))

# 7. Correlation and Feature Analysis

**Features:** `TV`, `Radio`, `Newspaper` | **Target:** `Sales`

How we judge which channels matter:
- **Correlation** shows how strongly each channel moves *together* with Sales, one channel at a time.
- **Linear Regression coefficients** show the estimated change in Sales per one-unit increase in a channel, *holding the other channels fixed*.
- **Random Forest feature importance** shows how much each feature helps the trees make accurate predictions (it can capture non-linear effects).

> ⚠️ **Association is not causation.** These tools show patterns in historical data. They do not prove that raising a channel's budget *causes* more sales. Hidden factors (seasonality, pricing, promotions) or how budgets were allocated could drive the patterns. Establishing causation would need controlled experiments.

# 8. Train/Test Split
We split **before** fitting any model or learned preprocessing (imputer, scaler). The test set stays untouched until final evaluation. 80% train / 20% test, `random_state=42`.

In [ ]:
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Training set: {X_train.shape[0]} rows | Test set: {X_test.shape[0]} rows")
print(f"Test share: {len(X_test) / len(X):.0%}")

# 9. Linear Regression Model (Baseline)
A pipeline bundles preprocessing and model. The imputer (median) only does something if features have missing values, and it is **fitted on the training data only**.

In [ ]:
def build_linear_model():
    """Baseline: median imputation (no-op when no NaNs) + Linear Regression."""
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LinearRegression()),
    ])

lin_model = build_linear_model()
lin_model.fit(X_train, y_train)
print("Linear Regression trained.")

# 10. Random Forest Regression Model
An ensemble of decision trees that can capture non-linear relationships and feature interactions. Trees do not need scaling.

In [ ]:
def build_rf_model():
    """Alternative: median imputation + Random Forest with fixed seed."""
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)),
    ])

rf_model = build_rf_model()
rf_model.fit(X_train, y_train)
print("Random Forest trained.")

# 11. Evaluation Metrics and Comparison

| Metric | Meaning | Better when |
|---|---|---|
| **MAE** | Average absolute prediction error, in Sales units | Lower |
| **RMSE** | Square root of the average squared error. Penalises large errors more | Lower |
| **R²** | Share of the variance in Sales explained by the model (1 = perfect, 0 = no better than predicting the mean) | Higher |

Both models are evaluated on the **same held-out test set**.

In [ ]:
def evaluate(model, X_eval, y_eval):
    """Return MAE, RMSE and R² for a fitted model."""
    pred = model.predict(X_eval)
    return {
        "MAE": mean_absolute_error(y_eval, pred),
        "RMSE": np.sqrt(mean_squared_error(y_eval, pred)),
        "R²": r2_score(y_eval, pred),
    }

models = {"Linear Regression": lin_model, "Random Forest": rf_model}
results = pd.DataFrame({name: evaluate(m, X_test, y_test) for name, m in models.items()}).T
print("Test-set performance:")
display(results)

In [ ]:
# Visual comparison of the calculated metrics
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric in zip(axes, ["MAE", "RMSE", "R²"]):
    sns.barplot(x=results.index, y=results[metric], ax=ax)
    ax.set_title(f"{metric} (test set)")
    ax.set_xlabel("")
    ax.set_ylabel(metric)
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.3f}", (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Supplementary robustness check: 5-fold cross-validation on the TRAINING data only
# (the test set is not touched). With ~200 rows, a single 40-row test set is noisy.
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = {}
for name, builder in [("Linear Regression", build_linear_model), ("Random Forest", build_rf_model)]:
    rmse_scores = -cross_val_score(builder(), X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error")
    r2_scores = cross_val_score(builder(), X_train, y_train, cv=cv, scoring="r2")
    cv_rows[name] = {"CV RMSE (mean)": rmse_scores.mean(), "CV RMSE (std)": rmse_scores.std(),
                     "CV R² (mean)": r2_scores.mean()}
cv_results = pd.DataFrame(cv_rows).T
print("5-fold cross-validation on training data (supporting evidence only):")
display(cv_results)

# 12. Best Model Selection
The best model is chosen from **test-set** results: lowest **RMSE** first, **MAE** as tie-breaker, then we check whether **R²** agrees. Nothing is hardcoded. The decision is computed from the table above.

In [ ]:
ranked = results.sort_values(["RMSE", "MAE"], ascending=True)
best_name = ranked.index[0]
best_model = models[best_name]

best_by_r2 = results["R²"].idxmax()
print(f"Best model (lowest RMSE, then MAE): {best_name}\n")
display(results.loc[[best_name]])

if best_by_r2 == best_name:
    print(f"R² agrees: {best_name} also has the highest R² ({results.loc[best_name, 'R²']:.4f}).")
else:
    print(f"Note: {best_by_r2} has the higher R², but {best_name} has lower prediction errors, "
          "which is our priority. The models are close on some metrics.")

other = [n for n in results.index if n != best_name][0]
print(f"\nCompared with {other}: RMSE is lower by {results.loc[other,'RMSE'] - results.loc[best_name,'RMSE']:.4f} "
      f"and MAE is lower by {results.loc[other,'MAE'] - results.loc[best_name,'MAE']:.4f}.")

# 13. Residual Analysis
Residual = actual - predicted. For a well-behaved model, residuals should scatter randomly around **zero** with no visible pattern or funnel shape.

In [ ]:
y_pred = best_model.predict(X_test)
residuals = y_test.values - y_pred

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# (a) Actual vs predicted
axes[0].scatter(y_test, y_pred, alpha=0.7)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, "r--", label="Perfect prediction")
axes[0].set_title(f"Actual vs Predicted Sales ({best_name})")
axes[0].set_xlabel("Actual Sales")
axes[0].set_ylabel("Predicted Sales")
axes[0].legend()

# (b) Residuals vs predicted with zero line
axes[1].scatter(y_pred, residuals, alpha=0.7)
axes[1].axhline(0, color="red", linestyle="--", label="Zero residual")
axes[1].set_title("Residual Plot")
axes[1].set_xlabel("Predicted Sales")
axes[1].set_ylabel("Residual (Actual - Predicted)")
axes[1].legend()

# (c) Distribution of residuals
sns.histplot(residuals, kde=True, ax=axes[2])
axes[2].axvline(0, color="red", linestyle="--")
axes[2].set_title("Distribution of Residuals")
axes[2].set_xlabel("Residual")

plt.tight_layout()
plt.show()

In [ ]:
# Numerical checks that back up the visual reading (computed, not assumed)
res_mean, res_std = residuals.mean(), residuals.std()
corr_res_pred = np.corrcoef(y_pred, residuals)[0, 1]
# Compare error size in the lower vs upper half of predictions to look for a funnel (heteroscedasticity)
order = np.argsort(y_pred)
half = len(order) // 2
low_sd, high_sd = residuals[order[:half]].std(), residuals[order[half:]].std()

print(f"Mean residual: {res_mean:+.4f}  (close to 0 means no systematic over/under-prediction)")
print(f"Residual std dev: {res_std:.4f}")
print(f"Correlation between residuals and predictions: {corr_res_pred:+.3f}")
print(f"Residual spread - lower half of predictions: {low_sd:.3f} | upper half: {high_sd:.3f}")

print("\nReading of the results:")
print("- Mean residual is", "close to zero relative to the spread." if abs(res_mean) < 0.25 * res_std
      else "noticeably away from zero, suggesting some systematic bias.")
print("- Residuals vs predictions are", "only weakly related (no strong trend)." if abs(corr_res_pred) < 0.3
      else "clearly related, so the model may be missing a pattern (for example non-linearity).")
print("- Spread is", "similar across the prediction range." if max(low_sd, high_sd) < 1.5 * min(low_sd, high_sd)
      else "uneven across the prediction range (possible funnel shape / non-constant variance).")
print("\nAlways confirm against the plots above: curves or funnels suggest the model misses non-linear effects or "
      "interactions, which would mean predictions are less reliable in those regions.")

# 14. Advertising Channel Impact

**Important caveats**
- Linear regression coefficients depend on each feature's **units/scale**, so raw coefficients are not strictly comparable across channels. We also show **standardised** coefficients (features scaled to mean 0 / std 1, fitted on the training data only).
- Correlated channels can share credit, which makes individual effects unstable.
- None of these measures proves **causal** impact.

In [ ]:
# --- Linear Regression: raw coefficients ---
lin_core = lin_model.named_steps["model"]
coef_df = pd.DataFrame({"Feature": FEATURES, "Coefficient": lin_core.coef_}).set_index("Feature")
print(f"Intercept: {lin_core.intercept_:.4f}")
print("\nRaw Linear Regression coefficients (change in Sales per +1 unit of spend, others fixed):")
display(coef_df)

# --- Linear Regression: standardised coefficients (scaler fitted on TRAIN only) ---
std_lin = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
]).fit(X_train, y_train)
coef_df["Standardised Coefficient"] = std_lin.named_steps["model"].coef_
print("\nStandardised coefficients (change in Sales per +1 std dev of spend):")
display(coef_df)

# --- Random Forest: impurity-based importance and permutation importance on the test set ---
rf_core = rf_model.named_steps["model"]
imp_df = pd.DataFrame({"Feature": FEATURES, "RF Importance": rf_core.feature_importances_}).set_index("Feature")
perm = permutation_importance(rf_model, X_test, y_test, n_repeats=30, random_state=RANDOM_STATE, scoring="r2")
imp_df["Permutation Importance (test, R² drop)"] = perm.importances_mean
print("\nRandom Forest feature importances:")
display(imp_df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

coef_df["Coefficient"].plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Linear Regression: Raw Coefficients")
axes[0].set_ylabel("Coefficient")
axes[0].axhline(0, color="black", linewidth=0.8)

coef_df["Standardised Coefficient"].plot(kind="bar", ax=axes[1], color="seagreen")
axes[1].set_title("Linear Regression: Standardised Coefficients")
axes[1].set_ylabel("Coefficient")
axes[1].axhline(0, color="black", linewidth=0.8)

imp_df["RF Importance"].plot(kind="bar", ax=axes[2], color="darkorange")
axes[2].set_title("Random Forest: Feature Importance")
axes[2].set_ylabel("Importance")

for ax in axes:
    ax.set_xlabel("Advertising channel")
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Plain-language summary generated from the actual numbers
std_coefs = coef_df["Standardised Coefficient"]
top_lin = std_coefs.abs().idxmax()
top_rf = imp_df["RF Importance"].idxmax()

print("Linear Regression (standardised):")
for ch, v in std_coefs.sort_values(key=abs, ascending=False).items():
    sign = "increases" if v > 0 else "decreases"
    print(f"  - {ch:<10} {v:+.3f} -> higher {ch} spend is associated with a model-estimated Sales that {sign} "
          f"(holding other channels fixed)")
print(f"\nMost influential channel per Linear Regression: {top_lin}")
print(f"Most influential channel per Random Forest:    {top_rf}")
print("Both models agree." if top_lin == top_rf else "The two models rank the top channel differently - interpret with care.")

# Flag any channel whose contribution looks negligible
weak = [ch for ch in FEATURES if imp_df.loc[ch, "RF Importance"] < 0.05 and abs(std_coefs[ch]) < 0.1 * std_coefs.abs().max()]
if weak:
    print(f"\nChannel(s) with very little apparent contribution once the others are accounted for: {weak}")
print("\nReminder: these are associations from model fits, not proof of causal effect.")

# 15. Sample Sales Prediction
We predict Sales for a **clearly labelled hypothetical** spending scenario using the selected best model. The input uses the same feature names and order as training. Edit the three numbers to try other scenarios. Keep them within the range seen in the data for reliable results.

In [ ]:
# Hypothetical example input (same units as the dataset's spend columns)
example_input = {"TV": 150.0, "Radio": 25.0, "Newspaper": 30.0}

sample_df = pd.DataFrame([example_input], columns=FEATURES)   # same names and order as training
predicted_sales = best_model.predict(sample_df)[0]

print("Example advertising budget (hypothetical):")
display(sample_df)
print(f"Model used: {best_name}")
print(f"Predicted Sales: {predicted_sales:.2f} (in the dataset's Sales units)")

# Safety check: warn if the example is outside the training range
for ch in FEATURES:
    lo, hi = X_train[ch].min(), X_train[ch].max()
    if not lo <= example_input[ch] <= hi:
        print(f"Warning: {ch}={example_input[ch]} is outside the training range [{lo:.1f}, {hi:.1f}]; "
              "prediction is an extrapolation and less reliable.")

# 16. Conclusion and Limitations

In [ ]:
# Final summary built from the actual results above
print("=" * 70)
print("FINAL SUMMARY (generated from this run's results)")
print("=" * 70)
print(f"Dataset: {df.shape[0]} rows | Train: {len(X_train)} | Test: {len(X_test)}")
print("\nTest-set metrics:")
print(results.round(4).to_string())
print(f"\nSelected model: {best_name} "
      f"(RMSE={results.loc[best_name,'RMSE']:.3f}, MAE={results.loc[best_name,'MAE']:.3f}, R²={results.loc[best_name,'R²']:.3f})")
print(f"Share of Sales variance explained on unseen data: {results.loc[best_name,'R²']:.1%}")
print(f"Most strongly correlated channel with Sales: {top}")
print(f"Most influential channel (Linear Regression, standardised): {top_lin}")
print(f"Most influential channel (Random Forest): {top_rf}")
print(f"Sample prediction for {example_input}: {predicted_sales:.2f}")

## Limitations
- **Small dataset:** about 200 rows means the 20% test set is small, so metrics can shift noticeably with a different split. The cross-validation table gives a steadier picture.
- **Association, not causation:** results describe patterns in historical data and do not prove that changing a channel's budget will change sales.
- **Multicollinearity and units:** correlated channels can share credit, and raw coefficients depend on units.
- **Limited features:** price, seasonality, competition, region and product details are not in the data.
- **Extrapolation risk:** predictions outside the observed spend ranges are unreliable (tree models cannot extrapolate beyond training values at all).
- **Light tuning:** Random Forest used sensible defaults, with no extensive hyperparameter search, to keep the notebook simple.

## Possible Next Steps
Try interaction terms (for example TV x Radio), gradient boosting, cross-validated hyperparameter tuning, and, if actionable decisions are needed, controlled marketing experiments.

# Internship Requirements Checklist

- [x] Libraries: Python, pandas, NumPy, scikit-learn, Matplotlib, Seaborn (Section 2)
- [x] Dataset source, download, path handling, column inspection and verification of `TV`, `Radio`, `Newspaper`, `Sales`; no fabricated data (Sections 3-4)
- [x] First five rows, shape, dtypes, missing values, duplicate count, descriptive statistics (Section 5)
- [x] Seaborn pairplot, three titled/labelled scatter plots with trend lines, annotated heatmap, interpretation (Section 6)
- [x] Features/target defined; correlation, coefficients and importance explained; association is not causation (Section 7)
- [x] `train_test_split` 80/20 with `random_state=42`, performed before learned preprocessing/modelling (Section 8)
- [x] Linear Regression baseline and Random Forest alternative with reproducible settings (Sections 9-10)
- [x] MAE, RMSE, R² on the same test set, comparison table with actual metrics, metric explanations (Section 11)
- [x] Best model chosen from actual test results (Section 12)
- [x] Actual vs predicted plot, residual plot with zero line, residual interpretation (Section 13)
- [x] Coefficients, standardised coefficients, feature importances, unit/causality caveats (Section 14)
- [x] Labelled sample prediction with consistent feature names and order (Section 15)
- [x] Conclusion based on actual results, limitations (Section 16)
- [x] Missing values handled without leakage (train-fitted imputer in pipeline); no data leakage
- [x] Notebook structure follows the 16 required sections